# Customer Churn Model using PyTorch

### Importing Dependencies

In [1]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#### Loading Custom Dataset

In [2]:
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'age':             np.random.randint(18, 70, n).astype(float),
    'income':          np.random.exponential(50000, n),
    'tenure_months':   np.random.randint(1, 120, n).astype(float),
    'monthly_charges': np.random.uniform(20, 120, n),
    'num_products':    np.random.randint(1, 5, n).astype(float),
    'contract_type':   np.random.choice(['Month-to-month','One year','Two year'], n),
    'payment_method':  np.random.choice(['Auto', 'Manual', 'Electronic'], n),
})
df.loc[np.random.choice(n, int(n*0.05), replace=False), 'age']    = np.nan
df.loc[np.random.choice(n, int(n*0.03), replace=False), 'income'] = np.nan

df['churn'] = (
    ((df['monthly_charges'] > 80) & (df['tenure_months'] < 200)) | 
    (df['contract_type'] == 'Month-to-month').astype(int)
)

print(f'Shape: {df.shape}')
print(f'Missing: {df.isnull().sum()}')
print(f'Class Imbalance: {df['churn'].value_counts(normalize = True).round(3).to_dict()}')

Shape: (200, 8)
Missing: age                10
income              6
tenure_months       0
monthly_charges     0
num_products        0
contract_type       0
payment_method      0
churn               0
dtype: int64
Class Imbalance: {True: 0.58, False: 0.42}


#### Splitting the data into train, test, val

In [3]:
feature_cols = ['age', 'income', 'tenure_months', 'monthly_charges', 'num_products', 
                'contract_type', 'payment_method']

X = df[feature_cols]
y = df.churn.values
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.15, random_state = 42, 
                                                  stratify = y_temp)
print(f'Split: Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

Split: Train: 144, Val: 26, Test: 30


#### Imputing and Scaling Data

In [4]:
num_features = ['age', 'income', 'tenure_months', 'monthly_charges', 'num_products']
cat_features = ['contract_type', 'payment_method']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('scaler', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

preprocessor.fit(X_train)

X_train_p = preprocessor.transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

n_features = X_train_p.shape[1]
print(f'Features after encoding: {n_features}')

Features after encoding: 11


#### Creating Tensors and Datasets

In [5]:
def make_dataset(X_np, y_np):
    X_t = torch.tensor(X_np, dtype = torch.float32)
    y_t = torch.tensor(y_np, dtype = torch.float32).unsqueeze(1)
    return TensorDataset(X_t, y_t)

train_dataset = make_dataset(X_train_p, y_train)
val_dataset = make_dataset(X_val_p, y_val)
test_dataset = make_dataset(X_test_p, y_test)

#### DataLoaders

In [6]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_dataset, batch_size = 128, shuffle = False, num_workers = 0)

#### Creating NN Model

In [7]:
class ChurnNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChurnNet(in_features = n_features).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

print(f'Total Parameters: {sum(p.numel() for p in model.parameters()):,}')

Total Parameters: 3,393


#### Creating Reusable Epoch runner for training, evaluation and testing

In [8]:
def run_epoch(model, loader, criterion, optimizer = None, device = 'cpu'):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for bX, by in loader:
            bX, by = bX.to(device), by.to(device)

            if is_training:
                optimizer.zero_grad()

            out = model(bX)
            loss = criterion(out, by)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * bX.size(0)
            preds = (torch.sigmoid(out) >= 0.5).float()
            correct += (preds == by).sum().item()
            total += bX.size(0)

    return total_loss / total, correct / total * 100

#### Training Loop with best-model

In [9]:
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(50):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_churn_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1:>3} | Train {train_loss:.4f} ({train_acc:.1f}%) | Val {val_loss:.4f} ({val_acc:.1f}%)')

Epoch  10 | Train 0.6469 (58.3%) | Val 0.6617 (57.7%)
Epoch  20 | Train 0.4896 (86.1%) | Val 0.5556 (69.2%)
Epoch  30 | Train 0.2143 (95.8%) | Val 0.3352 (84.6%)
Epoch  40 | Train 0.0721 (99.3%) | Val 0.1642 (92.3%)
Epoch  50 | Train 0.0324 (100.0%) | Val 0.0934 (96.2%)


#### Evaluating on Test Data

In [10]:
model.load_state_dict(torch.load('best_churn_model.pth'))
test_loss, test_acc = run_epoch(model, test_loader, criterion, None, device)
print(f'Test - Loss: {test_loss:.4f} | Accuracy {test_acc:.1f}%')

Test - Loss: 0.0285 | Accuracy 100.0%


#### Inferencing on New Customer Data

In [11]:
def predict_churn(model, preprocessor, raw_input: dict, device: str) -> dict:
    model.eval()
    input_df = pd.DataFrame([raw_input])
    input_p = preprocessor.transform(input_df)
    input_t = torch.tensor(input_p, dtype = torch.float32).to(device)

    with torch.no_grad():
        logit = model(input_t)
        prob = torch.sigmoid(logit).item()

    return {
        'churn_probability': round(prob, 4),
        'predicted_churn':   1 if prob >= 0.5 else 0,
        'confidence':        'High' if abs(prob - 0.5) > 0.3 else 
                            'Medium' if abs(prob - 0.5) > 0.1 else 'Low'
    }

result = predict_churn(model, preprocessor, {'age': 35.0, 'income': 45000.0, 'tenure_months': 8.0,
    'monthly_charges': 95.0, 'num_products': 1.0, 'contract_type': 'Month-to-month', 
    'payment_method': 'Manual'}, device)
print(f'New Customer Prediction: {result}')

New Customer Prediction: {'churn_probability': 0.9996, 'predicted_churn': 1, 'confidence': 'High'}
